In [ ]:
### Task 1: Understanding the Model Layers

#Answer these simple questions in your own words:

#1. What does the **Embedding** layer do to a word index?
"""converts each word index into a dense vector of 64 numbers (output_dim=64).
 Words with similar usage end up with similar vectors, and these vectors are learned 
 during training. Index 0 (padding) also gets a vector."""

#3. What is the role of the **Encoder LSTM**?
"""Encoder LSTM: reads the English sequence word by word and compresses the whole sentence
 into one fixed-size vector (64 values). LSTM(64) returns only its final output, which is
   what RepeatVector then copies."""

#5. What does the **Decoder LSTM** do?
"""Decoder LSTM: receives the repeated sentence vector at every output step and, with 
return_sequences=True, produces one output per French position. Each output is then used 
to pick a French word."""

#8. Why do we use `softmax` in the final layer?
"""Softmax: converts each position’s scores into probabilities over all French words, summing to 1.
 argmax then picks the most likely word, and sparse_categorical_crossentropy compares those 
 probabilities to the true word index."""

In [4]:
%pip install tensorflow pandas scikit-learn

  Using cached tensorflow-2.21.0-cp313-cp313-win_amd64.whl.metadata (4.5 kB)
  Using cached absl_py-2.5.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached gast-0.7.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached termcolor-3.3.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached grpcio-1.85.1-cp313-cp313-win_amd64.whl.metadata (3.9 kB)
  Using cached keras-3.15.1-py3-none-any.whl.metadata (6.4 kB)
  Using cached h5py-3.14.0-cp313-cp313-win_amd64.whl.metadata (2.7 kB)
  Using cached ml_dtypes-0.6.0-cp313-cp313-win_amd64.whl.metadata (8.8 kB)
  Using cached namex-0.1.0-py3-none-any.whl.metadata (322 bytes)
  Using cached optree-0.

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [13]:
import os, numpy as np, pandas as pd, tensorflow as tf, kagglehub
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

# 1. Load data
path = kagglehub.dataset_download("devicharith/language-translation-englishfrench")
csv = [f for f in os.listdir(path) if f.endswith(".csv")][0]
df = pd.read_csv(os.path.join(path, csv))
df.columns = ["eng", "fra"]
print(df.shape)

# 2. Keep short, common sentences so a small LSTM can learn them
df = df.dropna().drop_duplicates()
df["eng"] = df.eng.str.lower().str.strip()
df["fra"] = df.fra.str.lower().str.strip()
df = df[(df.eng.str.split().str.len() <= 5) & (df.fra.str.split().str.len() <= 7)]
df = df.sample(min(20000, len(df)), random_state=42)

# 3. Tokenize and pad
tok_eng = Tokenizer(); tok_eng.fit_on_texts(df.eng)
tok_frn = Tokenizer(); tok_frn.fit_on_texts(df.fra)
eng_pad = pad_sequences(tok_eng.texts_to_sequences(df.eng))
frn_pad = pad_sequences(tok_frn.texts_to_sequences(df.fra), padding="post")

eng_train, eng_test, frn_train, frn_test = train_test_split(eng_pad, frn_pad, test_size=0.2, random_state=42)
w_train = np.where(frn_train == 0, 0.1, 1.0)   # padding counts less
w_test = np.where(frn_test == 0, 0.1, 1.0)

# 4. Model (same architecture as the lab)
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(len(tok_eng.word_index) + 1, 128),
    tf.keras.layers.LSTM(128),
    tf.keras.layers.RepeatVector(frn_pad.shape[1]),
    tf.keras.layers.LSTM(128, return_sequences=True),
    tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(len(tok_frn.word_index) + 1, activation="softmax"))
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", weighted_metrics=["accuracy"])

# 5. Train
model.fit(eng_train, np.expand_dims(frn_train, -1), sample_weight=w_train,
          epochs=40, batch_size=128, validation_split=0.1)
print("Test:", model.evaluate(eng_test, np.expand_dims(frn_test, -1), sample_weight=w_test))

# 6. Translate
def translate(sentence):
    seq = pad_sequences(tok_eng.texts_to_sequences([sentence.lower()]), maxlen=eng_pad.shape[1])
    pred = np.argmax(model.predict(seq, verbose=0), axis=-1)
    return tok_frn.sequences_to_texts(pred)[0]

for s in ["How are you?", "I love you.", "Thank you.", "I am happy."]:
    print(s, "->", translate(s))

(175621, 2)
Epoch 1/40
113/113 ━━━━━━━━━━━━━━━━━━━━ 24s 192ms/step - accuracy: 0.1015 - loss: 3.6440 - val_accuracy: 0.1036 - val_loss: 3.3639
Epoch 2/40
113/113 ━━━━━━━━━━━━━━━━━━━━ 21s 189ms/step - accuracy: 0.1024 - loss: 3.3284 - val_accuracy: 0.1036 - val_loss: 3.3824
Epoch 3/40
113/113 ━━━━━━━━━━━━━━━━━━━━ 22s 197ms/step - accuracy: 0.1024 - loss: 3.3032 - val_accuracy: 0.1036 - val_loss: 3.3636
Epoch 4/40
113/113 ━━━━━━━━━━━━━━━━━━━━ 24s 215ms/step - accuracy: 0.1024 - loss: 3.2710 - val_accuracy: 0.1036 - val_loss: 3.3448
Epoch 5/40
113/113 ━━━━━━━━━━━━━━━━━━━━ 23s 204ms/step - accuracy: 0.1246 - loss: 3.2366 - val_accuracy: 0.1323 - val_loss: 3.2953
Epoch 6/40
113/113 ━━━━━━━━━━━━━━━━━━━━ 24s 208ms/step - accuracy: 0.1322 - loss: 3.1565 - val_accuracy: 0.1323 - val_loss: 3.2134
Epoch 7/40
113/113 ━━━━━━━━━━━━━━━━━━━━ 24s 216ms/step - accuracy: 0.1435 - loss: 3.0867 - val_accuracy: 0.1559 - val_loss: 3.1684
Epoch 8/40
113/113 ━━━━━━━━━━━━━━━━━━━━ 25s 224ms/step - accuracy: 0.15

In [14]:
for s in ["How are you?", "I love you.", "Where is the hotel?"]:
    print(s, "->", translate(s))

How are you? -> pourquoi vous vous vous vous
I love you. -> je vous
Where is the hotel? -> où est est ouverte
